# LLM Experimentation
Tests `route_query` and `generate_response` with sample natural-language expense questions.

In [6]:
# Load tokenizer
print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded!")

# Load model
print("Downloading and loading Qwen...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("\nQwen loaded successfully!")
print("Device:", model.device)
print("Data type:", model.dtype)

Loading tokenizer...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Qwen loaded successfully!
Device: cuda:0
Data type: torch.float16


In [14]:
import torch

system_prompt = """
You are the intent detection module of a Smart Expense Assistant.

Your job is to identify the user's request and extract information.
Do NOT calculate spending or invent expense records.

AVAILABLE INTENTS:
- calculate_spending
- compare_months
- highest_category
- categorize_expense
- show_expenses
- add_expense

ALLOWED EXPENSE CATEGORIES:
- Food
- Travel
- Shopping
- Utilities

STRICT RULES:

1. Select exactly one intent from the available intents.
2. Never invent an intent, category, amount, month or description.
3. Use null when information is missing.
4. Category must be one of the four allowed categories.
5. For categorize_expense, leave category as null.
   The separate classification model will determine it.
6. For spending queries, do not extract or invent a spending amount.
   The backend will calculate it.
7. Extract the first month into month.
8. For month comparisons, extract the second month into comparison_month.
9. Extract an expense amount only when the user explicitly provides it.
10. Extract the expense description only when the user provides one.
11. Return ONLY valid JSON. No explanations or markdown.

EXAMPLES:

User: How much did I spend on food in November?
Output:
{"intent":"calculate_spending","category":"Food","month":"November","comparison_month":null,"amount":null,"description":null}

User: Which category has the highest spending?
Output:
{"intent":"highest_category","category":null,"month":null,"comparison_month":null,"amount":null,"description":null}

User: Compare my expenses between October and November.
Output:
{"intent":"compare_months","category":null,"month":"October","comparison_month":"November","amount":null,"description":null}

User: Categorize this expense: Uber ride to office, 250 rupees.
Output:
{"intent":"categorize_expense","category":null,"month":null,"comparison_month":null,"amount":250,"description":"Uber ride to office"}

User: Show all my expenses in November.
Output:
{"intent":"show_expenses","category":null,"month":"November","comparison_month":null,"amount":null,"description":null}

User: Add a food expense of 300 rupees for lunch.
Output:
{"intent":"add_expense","category":"Food","month":null,"comparison_month":null,"amount":300,"description":"lunch"}
"""

test_queries = [
    "I spent 250 rupees on an Uber ride to office.",
    "Add a food expense of 300 rupees for lunch.",
    "I bought a new keyboard for 2500 rupees.",
    "How much did I spend on food last month?"
]

for query in test_queries:

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    print("Query:", query)
    print("Response:", response)
    print("-" * 60)

Query: I spent 250 rupees on an Uber ride to office.
Response: {"intent":"categorize_expense","category":"Utilities","month":"October","comparison_month":"November","amount":250,"description":"Uber ride to office"}
------------------------------------------------------------
Query: Add a food expense of 300 rupees for lunch.
Response: {"intent":"add_expense","category":"Food","month":"January","comparison_month":"February","amount":300,"description":"lunch"}
------------------------------------------------------------
Query: I bought a new keyboard for 2500 rupees.
Response: {"intent":"calculate_spending","category":"Utilities","month":"December","comparison_month":"January","amount":2500,"description":"keyboard"}
------------------------------------------------------------
Query: How much did I spend on food last month?
Response: {"intent":"calculate_spending","category":"Food","month":"last month","comparison_month":null,"amount":null,"description":"none"}
---------------------------

In [15]:
import torch

print("GPU:", torch.cuda.get_device_name(0))

total_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)

print(f"Total GPU memory: {total_memory:.2f} GB")
print(f"Allocated memory: {torch.cuda.memory_allocated() / (1024**3):.2f} GB")
print(f"Reserved memory: {torch.cuda.memory_reserved() / (1024**3):.2f} GB")

GPU: Tesla T4
Total GPU memory: 14.56 GB
Allocated memory: 0.93 GB
Reserved memory: 0.96 GB


In [16]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print("Loading tokenizer...")
tokenizer_15b = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")

model_15b = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model_15b.eval()

print("\nQwen 1.5B loaded successfully!")
print("Device:", model_15b.device)
print("Data type:", model_15b.dtype)

Loading tokenizer...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Qwen 1.5B loaded successfully!
Device: cuda:0
Data type: torch.float16


In [17]:
import torch

system_prompt = """
You are the intent detection module of a Smart Expense Assistant.

Your job is to identify the user's request and extract information.
Do NOT calculate spending or invent expense records.

AVAILABLE INTENTS:
- calculate_spending
- compare_months
- highest_category
- categorize_expense
- show_expenses
- add_expense

ALLOWED EXPENSE CATEGORIES:
- Food
- Travel
- Shopping
- Utilities

STRICT RULES:

1. Select exactly one intent from the available intents.
2. Never invent an intent, category, amount, month or description.
3. Use null when information is missing.
4. Category must be one of the four allowed categories.
5. For categorize_expense, leave category as null.
   The separate classification model will determine it.
6. For spending queries, do not extract or invent a spending amount.
   The backend will calculate it.
7. Extract the first month into month.
8. For month comparisons, extract the second month into comparison_month.
9. Extract an expense amount only when the user explicitly provides it.
10. Extract the expense description only when the user provides one.
11. Return ONLY valid JSON. No explanations or markdown.

EXAMPLES:

User: How much did I spend on food in November?
Output:
{"intent":"calculate_spending","category":"Food","month":"November","comparison_month":null,"amount":null,"description":null}

User: Which category has the highest spending?
Output:
{"intent":"highest_category","category":null,"month":null,"comparison_month":null,"amount":null,"description":null}

User: Compare my expenses between October and November.
Output:
{"intent":"compare_months","category":null,"month":"October","comparison_month":"November","amount":null,"description":null}

User: Categorize this expense: Uber ride to office, 250 rupees.
Output:
{"intent":"categorize_expense","category":null,"month":null,"comparison_month":null,"amount":250,"description":"Uber ride to office"}

User: Show all my expenses in November.
Output:
{"intent":"show_expenses","category":null,"month":"November","comparison_month":null,"amount":null,"description":null}

User: Add a food expense of 300 rupees for lunch.
Output:
{"intent":"add_expense","category":"Food","month":null,"comparison_month":null,"amount":300,"description":"lunch"}
"""

test_queries = [
    "I spent 250 rupees on an Uber ride to office.",
    "Add a food expense of 300 rupees for lunch.",
    "I bought a new keyboard for 2500 rupees.",
    "How much did I spend on food last month?"
]

for query in test_queries:

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query}
    ]

    inputs = tokenizer_15b.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model_15b.device)

    with torch.inference_mode():
        outputs = model_15b.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer_15b.eos_token_id
        )

    response = tokenizer_15b.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    print("Query:", query)
    print("Response:", response)
    print("-" * 60)

Query: I spent 250 rupees on an Uber ride to office.
Response: {"intent":"categorize_expense","category":null,"month":null,"comparison_month":null,"amount":250,"description":"Uber ride to office"}
------------------------------------------------------------
Query: Add a food expense of 300 rupees for lunch.
Response: {
  "intent": "add_expense",
  "category": "Food",
  "month": null,
  "comparison_month": null,
  "amount": 300,
  "description": "lunch"
}
------------------------------------------------------------
Query: I bought a new keyboard for 2500 rupees.
Response: {"intent":"add_expense","category":"Shopping","month":null,"comparison_month":null,"amount":2500,"description":"new keyboard"}
------------------------------------------------------------
Query: How much did I spend on food last month?
Response: {"intent":"calculate_spending","category":"Food","month":"last month","comparison_month":null,"amount":null,"description":null}
------------------------------------------------

In [2]:

import torch

system_prompt = """

You are the intent detection module of a Smart Expense Assistant.

Your task is to understand the user's request, identify all
required operations, arrange them in the correct execution
order, and extract the relevant information.

You must NOT calculate spending, classify expenses yourself,
or execute any operations.

AVAILABLE INTENTS:

1. calculate_spending
   - User wants to know how much they spent.
   - Extract category and month if provided.

2. compare_months
   - User wants to compare expenses between two months.
   - Extract the first month into month.
   - Extract the second month into comparison_month.

3. highest_category
   - User wants to identify the category with the highest
     spending.
   - Extract month if explicitly provided.

4. categorize_expense
   - Requests the separate ML classification model to
     determine the category of an expense.
   - Never determine the category yourself.

5. show_expenses
   - User wants to view their expense records.
   - Extract category and month if provided.

6. add_expense
   - Requests the backend to record a new expense.
   - Requires an expense description and amount when provided.
   - The backend handles the actual recording.

MULTIPLE INTENT RULES:

1. The output must contain an array called "intents".

2. The array must contain all operations required to
   fulfil the user's request.

3. Arrange intents in their required execution order.

4. For a request to add or record a new expense, ALWAYS
   return both intents in this exact order:

   [
       "categorize_expense",
       "add_expense"
   ]

5. The classification operation must complete before
   the expense is added.

6. For a request that only asks to classify an expense,
   return only:

   [
       "categorize_expense"
   ]

7. For analytical requests, return only the relevant
   analytical intent.

8. Do not add unnecessary or unrelated intents.

9. Never duplicate an intent in the array.

ALLOWED EXPENSE CATEGORIES:

- Food
- Travel
- Shopping
- Utilities

STRICT EXTRACTION RULES:

1. Never invent information or assume missing values.

2. Always return all six required fields, including intents.

3. Use JSON null for missing values.

4. Category must be one of the four allowed categories
   or null.

5. Never classify an expense yourself.
   The separate ML model handles classification.

6. For any request containing categorize_expense,
   category must be null.

7. For calculate_spending, amount must always be null.
   The backend performs the calculation.

8. Extract the first explicitly mentioned month or time
   period into month.

9. For compare_months, extract the second explicitly
   mentioned month into comparison_month.

10. Preserve relative time expressions such as
    "last month", "this month" and "last week".
    Do not calculate their actual dates.

11. Extract amount only when explicitly provided.
    Return it as a number without currency symbols.

12. Extract description only when an expense item or
    description is explicitly provided.

13. For analytical queries, description must be null.

14. Do not include category names or unnecessary
    explanations in the description.

15. Distinguish between classifying and recording:

    - Asking to classify an expense:
      categorize_expense

    - Asking to add, record or log an expense:
      categorize_expense followed by add_expense

16. If information is missing or ambiguous, use null
    instead of guessing.

OUTPUT RULES:

Return ONLY one valid JSON object.

Do not include explanations, Markdown, code fences,
or additional text.

REQUIRED JSON STRUCTURE:

{
    "intents": ["..."],
    "category": null,
    "month": null,
    "comparison_month": null,
    "amount": null,
    "description": null
}
"""


manager_queries = [
    "How much did I spend on food in November?",
    "Which category has the highest spending?",
    "Compare my expenses between October and November.",
    "Categorize this expense: Uber ride to office, ₹250.",
    "Show all my expenses in November.",
    "How much did I spend on shopping in October?"
]

for i, query in enumerate(manager_queries, 1):

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query}
    ]

    inputs = tokenizer_15b.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model_15b.device)

    with torch.inference_mode():
        outputs = model_15b.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer_15b.eos_token_id
        )

    response = tokenizer_15b.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    print(f"\nTEST {i}")
    print("Query:", query)
    print("Qwen Output:", response)
    print("-" * 70)


NameError: name 'tokenizer_15b' is not defined

In [19]:
import gc
import torch

# Remove references to the old model
del model
del tokenizer

# Clear unused GPU memory
gc.collect()
torch.cuda.empty_cache()

print("Old 0.5B model removed from memory!")

print(
    "GPU memory currently allocated:",
    round(torch.cuda.memory_allocated() / (1024**3), 2),
    "GB"
)

Old 0.5B model removed from memory!
GPU memory currently allocated: 2.88 GB


In [1]:
!git clone -b feature/ai-model-dev https://github.com/RaghuBharathi3/smart-expense-assistant-ml.git

Cloning into 'smart-expense-assistant-ml'...
fatal: could not read Username for 'https://github.com': No such device or address
